
## 1-2. Постановка задачи оптимальной остановки

Рассматривается задача оптимальной остановки, возникающая при оценке **американских опционов**, в частности американского опциона **put**.

В дискретном времени задача формулируется следующим образом:
для заданного стохастического процесса цены базового актива $S_t$ требуется найти оптимальный момент исполнения $\tau$, максимизирующий математическое ожидание дисконтированного выигрыша

$V_0 = \sup_{\tau \in \mathcal{T}} \mathbb{E}\left[e^{-r \tau} g(S_\tau)\right]$,
где $g(S) = (K - S)^+ $ — payoff американского put-опциона.

Ключевая сложность задачи заключается в том, что оптимальное решение требует сравнения **немедленного исполнения** с **продолжением** (continuation value), которое, в свою очередь, зависит от условного математического ожидания будущих выплат.

В рамках дипломной работы задача решается с использованием методов регрессии и нейронных сетей, а также анализируется возможность построения **хеджирующих стратегий**, основанных на найденной цене опциона и её производной по цене базового актива (дельте).

Классическим подходом к численному решению задачи оптимальной остановки является метод **Longstaff–Schwartz (LSM)**, основанный на аппроксимации continuation value с помощью линейной регрессии по базисным функциям от состояния.

Дальнейшее развитие литературы пошло по следующим направлениям:

- использование **регуляризации** и расширенных базисов (RLSM);
- применение **нейронных сетей** для аппроксимации conditional expectation (NLSM);
- переход к reinforcement learning и Q-learning методам (LSPI, FQI и др.);
- использование нейронных сетей, которые напрямую аппроксимируют цену и её градиенты, что открывает путь к хеджированию.

При этом в большинстве работ основной фокус делается на **оценке цены**, тогда как **хеджирование американских опционов** рассматривается значительно реже и зачастую в упрощённом виде.

Цель данной работы — воспроизвести и сравнить классические и нейросетевые методы оптимальной остановки **в едином экспериментальном окружении** и продемонстрировать, как нейронные методы естественным образом приводят к построению дельта-хеджа.

## 3. Описание используемых алгоритмов


### LSM — Least Squares Monte Carlo

Начнём с классического метода **LSM**, предложенного Longstaff и Schwartz.
Алгоритм основан на обратной индукции по времени. На каждом шаге мы сравниваем немедленное исполнение опциона с ожидаемой будущей стоимостью продолжения. Эта continuation value аппроксимируется с помощью линейной регрессии на заранее заданных базисных функциях от цены актива.

Обучение происходит **пошагово**, начиная с последнего момента времени и двигаясь назад к начальному моменту.
Основное преимущество LSM — его простота и надёжность: метод хорошо изучен, устойчив и широко используется как **эталон** для сравнения других алгоритмов.



### NLSM — Neural Least Squares Monte Carlo

Метод **NLSM** сохраняет общую структуру LSM — обратную индукцию и регрессию continuation value, — но заменяет линейную регрессию на **нейронную сеть**.

На каждом шаге времени нейронная сеть обучается аппроксимировать условное математическое ожидание будущей стоимости опциона. За счёт этого модель автоматически извлекает **сложные нелинейные зависимости** из данных, без явного задания базисных функций.

Обучение, как и в LSM, выполняется пошагово, но ключевым преимуществом NLSM является его **аппроксимационная мощность**. Кроме того, гладкость нейросетевой аппроксимации позволяет напрямую вычислять производные по цене актива, что делает метод особенно удобным для построения хеджирующих стратегий.



### RLSM — Randomized Least Squares Monte Carlo

Метод **RLSM** представляет собой модификацию LSM, в которой используются **рандомизированные признаки** вместо фиксированного базиса. По сути, continuation value аппроксимируется одной параметрической моделью, где скрытое представление формируется случайным образом и не обучается.

В отличие от LSM и NLSM, обучение здесь носит **глобальный характер**: модель сразу настраивается для всех сценариев и временных шагов. Это позволяет существенно ускорить вычисления и делает метод привлекательным с точки зрения масштабируемости.

Основное преимущество RLSM — **скорость** и простота реализации. Однако из-за отсутствия адаптации признаков метод может демонстрировать смещение в задачах с сложной границей оптимальной остановки.



## Итоговая интуиция для слушателя

> LSM — это надёжный классический метод.
> NLSM — его мощное нейросетевое обобщение, подходящее для сложных зависимостей и хеджирования.
> RLSM — быстрый и масштабируемый метод, эффективный в простых режимах, но чувствительный к качеству случайного представления.



## 4
RLSM демонстрирует более узкие доверительные интервалы, однако систематически занижает цену опциона из-за ограниченной выразительности случайного базиса. Нейросетевой метод NLSM обеспечивает наилучший баланс между смещением и дисперсией.


Наиболее узкий доверительный интервал наблюдается у NLSM, что говорит о меньшей дисперсии оценки при фиксированном числе траекторий.
Классический LSM демонстрирует наибольший разброс, что типично для линейной регрессии при увеличении сложности задачи оптимальной остановки.
Таким образом, нейросетевой метод оказывается более статистически устойчивым, несмотря на существенно более высокие вычислительные затраты на этапе обучения.

Очень сильная формулировка (можно почти дословно):
RLSM вводится не как более точный метод, а как вычислительно эффективная альтернатива LSM и NLSM в задачах с высокой размерностью и большим числом временных шагов.

Почему у RLSM возникает большее смещение (bias)
1. Неадаптивность базиса
Главевая причина смещения — базис не подстраивается под данные.
в LSM базис фиксирован, но подобран осмысленно;
в NLSM признаки обучаются градиентным спуском;
в RLSM признаки случайны и не оптимизируются.
В результате:
continuation value аппроксимируется неточно;
возникает систематическая ошибка.


2. Смещение вниз цены американского опциона
В задаче оптимальной остановки ошибка регрессии асимметрична:
недооценка continuation value ⇒ преждевременное исполнение;
преждевременное исполнение ⇒ потеря будущей ценности;
итог: смещение оценки цены вниз.
Это фундаментальное свойство LSM-подобных методов.


3. Почему доверительный интервал может быть узким
Важно понимать:
доверительный интервал отражает разброс между прогонами;
он не отражает смещение.
В RLSM:
случайный базис фиксируется;
регрессия становится стабильной;
variance снижается, но bias растёт.
Это классический bias–variance trade-off.



Поведение RLSM зависит от режима.

Когда RLSM работает плохо
1. Сложная граница остановки
Американские опционы:
имеют резкую границу исполнения;
чувствительны к ошибкам регрессии.
RLSM плохо аппроксимирует такие особенности.
2. Мало траекторий / много шагов
В вашем эксперименте:
50 шагов;
2000 траекторий;
ограниченное число random features.
Это неблагоприятный режим для RLSM.

## Сравнение сценариев Pricing (A / B / C)

| **Параметр**               | **Scenario A**<br>Base case  |
| -------------------------- | ---------------------------- |
| **Цель сценария**          | Базовое сравнение алгоритмов |
| **Модель актива**          | Black–Scholes                |
| **Payoff**                 | MinPut (1D put)              |
| **Число активов**          | 1                            |
| **Spot (S_0)**             | 100                          |
| **Strike (K)**             | 100                          |
| **Maturity (T)**           | 1.0                          |
| **Волатильность (\sigma)** | 0.2                          |
| **Drift (r-q)**            | 0.05                         |
| **Дивиденды**              | 0.0                          |
| **Число дат (n)**          | **50 (фикс.)**               |
| **Число траекторий (M)**   | **20 000 (фикс.)**           |
| **Число прогонов**         | 20                           |
| **Алгоритмы**              | LSM, RLSM, NLSM              |
| **train ITM only**         | True                         |
| **payoff как вход**        | True                         |
| **use_path**               | False                        |
| **Hidden size (NN)**       | 20                           |
| **Epochs (NN)**            | 30                           |
| **Что сравниваем**         | Среднюю цену и разброс       |
| **Тип ошибки**             | Общая                        |
| **Основной вывод**         | Методы согласованы           |





## 5
На графике показана зависимость средней цены американского put-опциона от числа временных шагов с 95% доверительными интервалами. Для всех алгоритмов доверительные интервалы остаются одного порядка, что говорит о стабильности оценки цены при изменении дискретизации времени. При этом нейросетевой метод демонстрирует наиболее ровное поведение средней цены, а у классического LSM наблюдается локальная нестабильность при среднем числе шагов.


RLSM даёт устойчиво более высокую оценку continuation value, а значит — более позднюю остановку и более высокую цену опциона.

## 6
Ниже — готовые формулировки «что сказать комиссии» **для каждой строки**, максимально ясно и без лишней математики, но при этом строго по смыслу.


## 1)$$\Delta = \partial V / \partial S$$

**Что это значит:**
Дельта — это чувствительность цены опциона (V) к изменению цены базового актива (S).

«Дельта показывает, насколько изменится цена опциона при малом изменении цены акции. Это базовый коэффициент для дельта-хеджирования: сколько единиц базового актива нужно держать, чтобы компенсировать движение цены опциона.»


## 2) $$X_1 = V_0 + \Delta_0 (S_1 - S_0)$$

**Что это значит:**
Это стоимость **хедж-портфеля через один шаг времени**. Мы стартуем с капиталом (X_0=V_0) и берём позицию (\Delta_0) в базовом активе. Если актив изменился с (S_0) до (S_1), портфель меняется на (\Delta_0(S_1-S_0)).

«В момент времени 0 я беру цену опциона (V_0) как стартовый капитал и открываю позицию (\Delta_0) в базовом активе. Через один временной шаг получаю новую стоимость портфеля: начальная стоимость плюс прибыль/убыток от позиции в активе. Это и есть one-step ребалансировка.»

*(Если вас спросят про процентную ставку: можно добавить, что в упрощённой версии кэш не начисляет проценты; при необходимости учитывается фактор (e^{r\Delta t}).)*



## 3) $$L = (V_1 - X_1)_+$$

**Что это значит:**
Это **shortfall (недохедж)** — величина, показывающая, насколько не хватает средств портфеля, чтобы «покрыть» стоимость опциона через один шаг.

* Если (X_1 \ge V_1), то (L=0) (хедж успешен).
* Если (X_1 < V_1), то (L = V_1 - X_1) (портфель не дотянул).

«Я измеряю качество хеджа через shortfall — дефицит средств. Меня интересуют именно плохие сценарии, когда денег не хватает. Поэтому беру положительную часть ошибки: если портфель перекрыл опцион — это не проблема; проблема только когда не хватает.»

---

## 4) $$\mathrm{VaR}*{0.99}(L),\ \mathrm{CVaR}*{0.99}(L)$$

**Что это значит:**
Это **tail-метрики риска** по распределению shortfall (L), рассчитанному на множестве траекторий.

* (\mathrm{VaR}_{0.99}(L)): 99-й квантиль shortfall — «порог» потерь, который превышается примерно в 1% худших случаев.
* (\mathrm{CVaR}_{0.99}(L)): средний shortfall **внутри худших 1% случаев** (expected shortfall), то есть более строгая мера tail-риска.

«Дальше я не просто смотрю среднюю ошибку, а оцениваю риск в хвосте распределения. VaR на 99% — это порог недохеджа в худших сценариях, а CVaR — средний недохедж именно в самых плохих 1% траекторий. Это и есть количественный результат one-step hedge: насколько стратегия опасна в редких, но тяжёлых ситуациях.»


«Дельта задаёт позицию в базовом активе, по ней я строю портфель на один шаг, считаю недохедж shortfall и измеряю хвостовой риск shortfall через VaR/CVaR на уровне 99%.»
---


## *Hedging risk sensitivity to discretization (VaR)*

### Что изображено

* **X:** `nb_dates`
* **Y:** VaR_{0.99} shortfall относительно стратегии buy-and-hold
* Линии — LSM / NLSM / RLSM

Это **риск хедж-ошибки на один шаг**.

### Что мы видим

#### Общий тренд

* Для всех методов:

  * VaR **растёт** при увеличении `nb_dates`.

**Интерпретация:**

> Более мелкая временная сетка делает one-step hedge более чувствительным к ошибкам аппроксимации.


#### Сравнение методов

##### NLSM

* **Самый низкий VaR** при всех `nb_dates`
* Рост плавный и монотонный.

**Интерпретация:**

> Аналитическая дельта, получаемая из нейросети, даёт более устойчивый хедж.

##### LSM

* VaR выше, чем у NLSM;
* монотонный рост.

**Интерпретация:**

> Численная дельта усиливает шум, особенно при более сложной временной структуре.

##### RLSM

* Немонотонное поведение:

  * резкий скачок при `nb_dates = 50`.

**Интерпретация:**

> Регуляризация улучшает pricing, но может ухудшать локальные свойства дельты, что критично для хеджирования.


### Корректный вывод по графику 2

> Хотя RLSM стабилизирует цену, это не гарантирует улучшения хедж-риска. NLSM демонстрирует наименьший VaR при всех уровнях дискретизации.


# График

## *Hedging risk sensitivity to discretization (CVaR)*

### Что изображено

* **X:** `nb_dates`
* **Y:** CVaR_{0.99} shortfall
* Те же три алгоритма.

CVaR — **средний размер наихудших потерь**, более строгая мера риска, чем VaR.


### Что мы видим

#### Общий тренд

* CVaR растёт с увеличением `nb_dates` для всех методов.

**Интерпретация:**

> Tail-риск увеличивается при более сложной временной структуре задачи.

#### Сравнение методов

##### NLSM

* Самый низкий CVaR при всех `nb_dates`
* Рост почти линейный.

**Интерпретация:**

> Нейросетевой метод минимизирует tail-риск хеджирования.


##### LSM

* CVaR стабильно выше, чем у NLSM.

**Интерпретация:**

> Ошибки численной дельты усиливаются в хвостах распределения.


##### RLSM

* Максимальный CVaR при `nb_dates = 50`.

**Интерпретация:**

> Регуляризация continuation value может приводить к искажённой дельте в экстремальных сценариях.


### Корректный вывод по графику 3

> В терминах tail-риска нейросетевой метод существенно превосходит классические регрессионные подходы.

### Ключевые подтверждения

* **NLSM**:

  * минимальные `VaR_mean` и `CVaR_mean`;
  * минимальные стандартные отклонения → высокая воспроизводимость.

* **RLSM**:

  * иногда выше LSM по CVaR;
  * больший риск в хвостах.

* **LSM**:

  * средний уровень риска, но более шумный.


# 5. Главный сквозной вывод (очень важен)


> «Регуляризация и нейросетевые методы улучшают устойчивость pricing, однако только нейросетевой метод, благодаря аналитической дельте, даёт систематическое улучшение хедж-риска. Это показывает, что хорошая аппроксимация цены не гарантирует хорошего хеджа.»



## 10 Дальнейшие планы работы

### 1) Переход от one-step hedge к многошаговому динамическому хеджированию

Сейчас оценка качества хеджа сделана на **один шаг** ([0,\Delta t]). Следующий этап — построить **полную стратегию ребалансировки**:
[
X_{t_{k+1}} = X_{t_k} + \Delta_{t_k}(S_{t_{k+1}}-S_{t_k})
]
(при желании — с учётом безрисковой ставки и хранения кэша).
Это позволит оценивать не «локальную» ошибку, а **итоговую P&L и риск на всём горизонте**.

**Что измерять:** распределение конечного shortfall, VaR/CVaR по конечному горизонту, вероятность недохеджа.

### 4) Проверка чувствительности к параметрам модели рынка

В текущих экспериментах (Black–Scholes) параметры фиксированы. Дальше:

* варьировать волатильность (\sigma), процентную ставку (r), страйк (K), maturity (T);
* построить карты чувствительности: **цена и риск хеджа vs параметры**.

**Цель:** понять, где методы устойчивы, а где деградируют.

Тестирование на реальных данных
---


### 3) Нормировка метрик риска и интерпретируемость результатов

Поскольку абсолютные VaR/CVaR выглядят большими, полезно добавить:

* ( \mathrm{VaR}/V_0 ), ( \mathrm{CVaR}/V_0 ) (в долях цены опциона),
* ( \mathrm{VaR}/S_0 ) (в долях цены базового),
* а также медиану/квантили shortfall.

**Цель:** сделать выводы интуитивными для комиссии и сопоставимыми между сценариями.



### 5) Расширение модельной среды: Heston и более реалистичная динамика

Логичное продолжение — перенести эксперименты на **Heston** (стохастическая волатильность):

* сравнить pricing и hedging при одном и том же числе траекторий/шагов;
* посмотреть, насколько NLSM сохраняет преимущество по хвостовому риску.

**Цель:** показать применимость вне Black–Scholes и повысить практическую ценность результатов.

---

### 6) Улучшение хеджа: учёт гаммы и регуляризация дельты

One-step delta hedge чувствителен к нелинейности (гамме). Возможные улучшения:

* оценивать (\Gamma = \partial^2 V/\partial S^2) (особенно естественно для NLSM),
* тестировать **delta-gamma hedge** (при наличии второго инструмента или аппроксимации),
* либо вводить **сглаживание/регуляризацию дельты**, чтобы уменьшить шум и tail-риск.

**Цель:** снизить VaR/CVaR именно в хвостах.

---

### 7) Добавление рыночных фрикций и реалистичности стратегии

Чтобы приблизиться к прикладной постановке:

* учесть транзакционные издержки при ребалансировке,
* ограничение на частоту торговли,
* возможные ограничения на позицию ((\Delta)-лимиты).

**Цель:** проверить, сохраняется ли преимущество NLSM в более реалистичной среде.

---

### 8) Бенчмарки и валидация: сравнение с эталонами

Для качества работы полезно добавить контрольные точки:

* для European put: сравнение с формулой Black–Scholes (цена и дельта),
* для American put: сравнение с численным эталоном (например, binomial tree / finite differences / Longstaff–Schwartz на очень большом числе путей).

**Цель:** отделить ошибку алгоритма от статистической ошибки Монте-Карло.

### 2) Явное разделение train/inference и ускорение для NLSM

По runtime-графикам видно, что NLSM дорог по времени. В планах:

* отделить **время обучения** от **времени применения** (inference);
* обучать **одну модель на все даты** (multi-output или time-embedding), а не фактически «локально» на каждом шаге;
* рассмотреть раннюю остановку, уменьшение архитектуры, батчирование.

**Цель:** получить сопоставимый риск хеджа при меньшей вычислительной стоимости.

## 
на слайде (5–6 буллетов)

* Многошаговый динамический хедж и итоговый риск на горизонте (T)
* Разделение train/inference, ускорение NLSM и единая модель по времени
* Нормированные метрики риска: VaR/CVaR в долях (V_0) и (S_0)
* Чувствительность к параметрам рынка ((\sigma, r, K, T))
* Расширение на Heston и стресс-сценарии
* Учёт фрикций: комиссии, дискретная торговля, ограничения на позиции

-


## 4. Влияние параметров дискретизации и вычислительная сложность

В рамках численных экспериментов исследовалось влияние:

- числа временных шагов ( N );
- количества траекторий;
- размерности состояния;
- архитектуры нейронной сети (для NLSM).

### Наблюдения:

- при увеличении числа временных шагов **дисперсия оценки цены возрастает**, особенно для LSM;
- NLSM демонстрирует более стабильные интервалы цен при мелкой дискретизации;
- время обучения:
    - LSM / RLSM — быстрое обучение, но на каждом новом наборе параметров;
    - NLSM — более дорогое обучение, но дешёвый inference;
- inference для NLSM практически не зависит от числа траекторий.

Таким образом, нейронные методы оказываются особенно выгодными в сценариях многократного использования модели.

RLSM демонстрирует более узкие доверительные интервалы, однако систематически занижает цену опциона из-за ограниченной выразительности случайного базиса. Нейросетевой метод NLSM обеспечивает наилучший баланс между смещением и дисперсией.



## 5. Результаты хеджирования на один шаг

Так как на каждом моменте времени была обучена модель цены американского опциона, в момент времени ( t = 0 ) доступны:

- оценка цены опциона ( V_0 ) всеми тремя методами;
- дельта:
    - для LSM — численно;
    - для RLSM и NLSM — аналитически как производная по ( S ).

Для каждой траектории строится **один шаг хеджирования**:

$$

X_1 = X_0 + \Delta_0 (S_1 - S_0),

$$

где  $X_0 = V_0$ .

Далее величина $X_1$  сравнивается с истинной ценой американского опциона в момент  $\Delta t$ .

Анализируются:

- среднее хедж-ошибки;
- стандартное отклонение;
- Value-at-Risk (VaR).



### Основные выводы:

- NLSM даёт более стабильный хедж;
- численная дельта в LSM приводит к большей дисперсии ошибки;
- нейросетевой подход естественным образом объединяет pricing и hedging.


## 6. Дальнейшие планы

В качестве продолжения работы планируется:

- расширение анализа на **многошаговый динамический хедж**;
- применние моделей на реальных данных;
- исследование моделей с стохастической волатильностью (Heston);
- анализ устойчивости нейросетевых дельт к изменению параметров рынка;



Payoff (Выплата) — это немедленная прибыль, которую инвестор получает при непосредственном исполнении американского опциона в конкретный момент времени и при конкретной цене базового актива. Это значение, которое сравнивается с продолжающейся стоимостью, чтобы принять решение: исполнять опцион сейчас или ждать.

Регуляризация в RLSM стабилизирует регрессию, но в задаче оптимальной остановки она приводит к систематическому занижению continuation value. Это смещение усиливается за счёт операции максимума и приводит к более раннему исполнению опциона, что объясняет более низкую цену по сравнению с LSM и NLSM.